# Evaluating Grounding: Measuring Faithfulness in Advanced RAG Systems

In Retrieval-Augmented Generation (RAG), the primary goal is to ensure that the LLM's response is not merely plausible, but verifiably supported by the provided source documents. This concept of **faithfulness**—the degree to which every claim made in the generated answer can be traced back to the retrieved context—is arguably the most critical metric for building reliable production-grade RAG systems. When an LLM hallucinates or introduces unsupported facts, the entire system fails its core mission: providing accurate, grounded information.

As we move into advanced architectures like LangGraph, where complex multi-step reasoning and tool use are common, maintaining high faithfulness becomes exponentially harder. A failure in grounding at one step can cascade, leading to catastrophic errors downstream. This notebook introduces `ragas`, a powerful framework for evaluating RAG pipelines by quantifying faithfulness. By running these metrics, developers gain the ability to systematically identify weak points—such as overly creative or unconstrained LLM prompts—and implement guardrails (e.g., self-correction loops in LangGraph) that force the model to adhere strictly to the provided context.

By mastering faithfulness evaluation, you transition from simply building a working RAG system to engineering a *trustworthy* one. You will learn how to quantitatively measure hallucination risk and use these scores to iteratively improve your retrieval components, prompt design, and overall pipeline robustness, ensuring that every answer delivered is factually accountable to its source material.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Define Faithfulness:** Explain the concept of grounding in RAG and understand why it is a critical metric for production systems.
*   **Utilize `ragas`:** Implement the `Faithfulness` metric using the `ragas` library to programmatically score LLM outputs against source contexts.
*   **Analyze Grounding Failures:** Interpret varying faithfulness scores (e.g., high vs. low) and diagnose whether a response is hallucinating or merely extrapolating beyond the context.
*   **Improve RAG Reliability:** Understand how quantitative evaluation of faithfulness informs advanced pipeline design, such as implementing verification steps within LangGraph to minimize hallucinations.


### Code Explanation

This cell initializes the necessary components for calculating faithfulness. It sets up an asynchronous OpenAI client, uses `llm_factory` to instantiate a specific LLM (here, 'gpt-5-mini'), and finally creates a `Faithfulness` scorer object using this LLM.


In [1]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

# Initialize the asynchronous OpenAI client for API calls
client = AsyncOpenAI()
# Use llm_factory to create an LLM instance (e.g., 'gpt-5-mini') using the configured client
llm = llm_factory("gpt-5-mini", client=client)
# Instantiate the Faithfulness scorer, passing the initialized LLM object to it
scorer = Faithfulness(llm=llm)


d:\rag-evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This cell demonstrates the evaluation of 'faithfulness,' a critical metric in RAG systems. It uses `scorer.ascore` to check if the generated response is accurately supported by the provided context, specifically flagging unsupported claims (like claiming prevention of all cancers).


In [2]:
# Example 1: Response is mostly grounded but adds one unsupported claim
# 'has been proven to prevent all forms of cancer' is not in the context
result = await scorer.ascore(
    user_input="What are the health benefits of green tea?",
    response="Green tea contains antioxidants that help reduce inflammation. It also boosts metabolism and has been proven to prevent all forms of cancer.",
    retrieved_contexts=[
        "Green tea is rich in antioxidants, particularly catechins, which help reduce inflammation and oxidative stress.",
        "Studies suggest green tea may modestly boost metabolic rate."
    ]
)
print(f"Faithfulness Score: {result.value}")


Faithfulness Score: 0.5


### Code Explanation (Faithfulness Scoring)

This cell demonstrates how to calculate the 'faithfulness' score, which measures whether every claim made in the generated response is directly supported by the provided context. It uses an asynchronous call (`await scorer.ascore`) on a pre-initialized `scorer` object.


In [3]:
# Example 2 : Every claim in the response is directly supported by the context
result = await scorer.ascore(
    user_input="When was the first Super Bowl played?",
    response="The first Super Bowl was played on January 15, 1967, at the Los Angeles Memorial Coliseum.",
    retrieved_contexts=[
        "The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles, California."
    ]
)
print(f"Faithfulness Score: {result.value}")


Faithfulness Score: 1.0


This cell demonstrates a failure case for faithfulness scoring. It uses the `scorer` to evaluate a generated response against limited context, specifically highlighting instances where the model 'hallucinates' facts (e.g., travel time or Earth-circling claims) not present in the provided retrieved documents.


In [4]:
# Example 3: Response introduces multiple facts not found in the context
# Context only states the speed; travel time to Earth and Earth-circling claim are hallucinated
result = await scorer.ascore(
    user_input="What is the speed of light?",
    response="The speed of light is approximately 3x10^8 meters per second. It takes light about 8 minutes to travel from the Sun to Earth, and light can circle the Earth 7.5 times in one second.",
    retrieved_contexts=[
        "The speed of light in a vacuum is approximately 299,792,458 meters per second."
    ]
)
print(f"Faithfulness Score: {result.value}")


Faithfulness Score: 0.3333333333333333
